In [1]:
# 05_multimetric_monitoring.ipynb
#
# Experiment 5 -- Multi-metric monitoring and the impossibility trade-off.
#
# Purpose:
#   Show that optimizing one fairness metric (DIR via group-wise thresholds) does
#   NOT jointly satisfy equalized odds, and that the direction of the inter-metric
#   interaction is domain-dependent (it can widen OR shrink the equalized-odds gap).
#   Compares baseline vs governed on DIR, dTPR, dFPR across the synthetic generator,
#   German (age), and Taiwan (sex) -- so the trade-off is shown on real data, not
#   only synthetic. 30 seeds, 95% CIs.
#
# Outputs (PNG+PDF, 600 dpi, grayscale, no captions; paths not printed):
#   results/tables/exp5_multimetric.csv
#   results/figures/exp5_multimetric.(png|pdf)

import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for _d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(_d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85"})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def dir_ratio(y_pred, group):
    approve=(y_pred==0).astype(int)
    r1=approve[group==1].mean(); r0=approve[group==0].mean()
    return (r1/r0) if r0>0 else np.nan
def eo_gaps(y_true,y_pred,group):
    def rate(ct,mask):
        idx=(y_true==ct)&mask
        return (y_pred[idx]==1).mean() if idx.sum() else np.nan
    a,b=group==1,group==0
    return abs(rate(1,a)-rate(1,b)), abs(rate(0,a)-rate(0,b))

def gen_synth(seed,n=6000):
    rng=np.random.default_rng(seed)
    A=rng.binomial(1,0.35,n)
    region=1.4*A+rng.normal(0,1,n); spending=1.4*A+rng.normal(0,1,n)
    income=rng.normal(0,1,n); debt=rng.normal(0,1,n); payhist=rng.normal(0,1,n)
    tl=0.9*income-0.8*debt+0.7*payhist+rng.normal(0,0.8,n)
    td=(tl<np.quantile(tl,0.30)).astype(int)
    flip=(A==1)&(td==0)&(rng.random(n)<0.9*0.25)
    X=pd.DataFrame({"region":region,"spending":spending,"income":income,"debt":debt,"payhist":payhist})
    return X, np.where(flip,1,td), A

def load_german():
    df=pd.read_csv(os.path.join(DATA_DIR,"german_credit.csv"))
    y=(df["credit_risk"]==0).astype(int).values; g=(df["age"]<25).astype(int).values
    cat=[c for c in df.select_dtypes(include=["object","string"]).columns]
    X=pd.get_dummies(df.drop(columns=["credit_risk","age"]),columns=cat).astype(float).reset_index(drop=True)
    return X,y,g

def load_taiwan():
    df=pd.read_parquet(os.path.join(DATA_DIR,"taiwan_credit.parquet"))
    y=df["DEFAULT"].astype(int).values; g=(df["SEX"]==2).astype(int).values
    drop=["DEFAULT","VIP","SEX_STD","AGE_YEARS","SEX","AGE","MARRIAGE"]
    X=df.drop(columns=[c for c in drop if c in df.columns]).astype(float).reset_index(drop=True)
    return X,y,g

def run(X,y,g,seed,gov,subs=None):
    idx=np.arange(len(y))
    if subs and len(y)>subs: idx=np.random.default_rng(seed).choice(idx,subs,replace=False)
    Xs=X.iloc[idx].reset_index(drop=True); ys=y[idx]; gs=g[idx]
    Xtr,Xte,ytr,yte,gtr,gte=train_test_split(Xs,ys,gs,test_size=0.3,random_state=seed,stratify=ys)
    sc=StandardScaler().fit(Xtr)
    m=LogisticRegression(max_iter=2000).fit(sc.transform(Xtr),ytr)
    p=m.predict_proba(sc.transform(Xte))[:,1]
    if not gov: yp=(p>=0.5).astype(int)
    else:
        br=((p>=0.5).astype(int)==0).mean()
        t1=np.quantile(p[gte==1],br); t0=np.quantile(p[gte==0],br)
        yp=np.where(gte==1,(p>=t1),(p>=t0)).astype(int)
    dt,dfp=eo_gaps(yte,yp,gte)
    return dir_ratio(yp,gte),dt,dfp

SEEDS=list(range(30))
SETS=[("synthetic",lambda s:gen_synth(s),None),
      ("german",None,None),("taiwan",None,8000)]
Xg,yg,gg=load_german(); Xt,yt,gt=load_taiwan()
rows=[]
for name,_,subs in SETS:
    for gov in [False,True]:
        vals=[]
        for s in SEEDS:
            if name=="synthetic": X,y,g=gen_synth(s)
            elif name=="german":  X,y,g=Xg,yg,gg
            else:                 X,y,g=Xt,yt,gt
            vals.append(run(X,y,g,s,gov,subs))
        arr=np.array(vals); m=arr.mean(0); ci=1.96*arr.std(0)/np.sqrt(len(SEEDS))
        rows.append({"dataset":name,"engine":"governed" if gov else "baseline",
                     "DIR":m[0],"DIR_ci":ci[0],"dTPR":m[1],"dTPR_ci":ci[1],"dFPR":m[2]})
res=pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR,"exp5_multimetric.csv"),index=False)
print(res.round(3).to_string(index=False))

# Grouped bars: DIR and dTPR, baseline vs governed, per dataset
sets=["synthetic","german","taiwan"]
fig,axes=plt.subplots(1,2,figsize=(10,4.2))
for ax,metric,lab in [(axes[0],"DIR","Disparate Impact Ratio (higher fairer)"),
                       (axes[1],"dTPR","Equalized-odds gap dTPR (lower fairer)")]:
    base=[res[(res.dataset==d)&(res.engine=="baseline")][metric].values[0] for d in sets]
    gov =[res[(res.dataset==d)&(res.engine=="governed")][metric].values[0] for d in sets]
    x=np.arange(len(sets))
    ax.bar(x-0.2,base,0.4,label="baseline",color=GRAYS[0],edgecolor="black",linewidth=1.0)
    ax.bar(x+0.2,gov,0.4,label="governed",color=GRAYS[2],edgecolor="black",linewidth=1.0)
    ax.set_xticks(x); ax.set_xticklabels(sets); ax.set_ylabel(lab)
    ax.legend(frameon=True,edgecolor="black")
fig.tight_layout(); save_fig(fig,"exp5_multimetric"); plt.close(fig)
print("Done: multi-metric monitoring across synthetic + 2 real datasets.")


  dataset   engine   DIR  DIR_ci  dTPR  dTPR_ci  dFPR
synthetic baseline 0.873   0.011 0.111    0.015 0.081
synthetic governed 1.000   0.000 0.225    0.007 0.014
   german baseline 0.835   0.033 0.102    0.027 0.137
   german governed 0.992   0.003 0.170    0.032 0.052
   taiwan baseline 1.021   0.005 0.032    0.009 0.012
   taiwan governed 1.000   0.000 0.042    0.008 0.005


Done: multi-metric monitoring across synthetic + 2 real datasets.
